# MapWorkIds — location_work_ids maintenance (oxjob #764)

Nightly identity step between `Locations_with_Types` and `Locations_Mapped`.
**Resolve-before-mint**: pending anchors are resolved against the existing
`work_id_map` FIRST, then new map rows are inserted already carrying their verdict —
an explicit id (converged/adopted) or a generated one (genuinely new work). Rows are
born correct, so no post-insert id correction exists; `work_id_map.id` is the single
verdict column (corrected only by audited delete+reinsert sweeps). Resolution precedes
minting, so every map row the resolve sees predates this run — no timestamp bookkeeping
for "established", and midnight-straddling runs are safe by construction.
doi matching is on the CLEANED form (`[^a-zA-Z0-9./-]` stripped) on both sides —
raw variants of one cleaned doi converge. Registry discipline unchanged: insert-only;
pinned anchors are never re-resolved.

In [ ]:
SELECT CASE WHEN COUNT(*) > 0 THEN RAISE_ERROR(
  'work_id_map still has the legacy work_id column — this notebook requires the '
  || 'single-id schema. Run qa/oxjob764_map_single_id_migration.py before deploying.')
END AS migration_guard
FROM openalex.information_schema.columns
WHERE table_schema = 'works' AND table_name = 'work_id_map' AND column_name = 'work_id'

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.work_id_map (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 6600000001 INCREMENT BY 1),
  doi STRING,
  pmid STRING,
  arxiv STRING,
  title_author STRING,
  created_date DATE,
  updated_date TIMESTAMP
)
CLUSTER BY (doi, pmid, arxiv, title_author)
TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = '60 days',
  'delta.logRetentionDuration' = '60 days',
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.location_work_ids (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  work_id BIGINT,
  work_id_source STRING,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  seeded_dt DATE,
  seeded_from STRING
)
CLUSTER BY (provenance, native_id_namespace, native_id)

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.location_work_ids_run_stats (
  run_ts TIMESTAMP,
  run_dt DATE,
  anchors_new_resolved BIGINT,
  anchors_new_null BIGINT,
  anchors_retried_resolved BIGINT,
  anchors_retried_null BIGINT,
  src_doi BIGINT,
  src_pmid BIGINT,
  src_arxiv BIGINT,
  src_title_author BIGINT,
  src_mag_legacy BIGINT,
  src_pmh_legacy BIGINT,
  minted_range BIGINT,
  distinct_works BIGINT,
  max_work_fanin BIGINT,
  top_fanin_work_id BIGINT,
  registry_null_backlog BIGINT,
  mints_displaced BIGINT,
  bridge_conflicts BIGINT,
  converged_alias BIGINT
)

## Pending anchors
One row per anchor needing a verdict: absent from the registry, or registered NULL
(NULL anchors retry nightly). Keys are scrubbed of degenerate values here, once, and
the cleaned doi is computed here, once — every later cell reads this table, so mint
and resolve can never disagree about the pending universe or the key forms.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.location_work_ids_pending
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM openalex.works.locations_w_types
  QUALIFY rwcnt = 1
)
SELECT t.provenance, t.native_id_namespace, t.native_id,
       (r.native_id IS NOT NULL) AS is_retry,
       NULLIF(t.merge_key.doi, '')  AS doi,
       NULLIF(regexp_replace(t.merge_key.doi, '[^a-zA-Z0-9\./-]', ''), '') AS doi_clean,
       NULLIF(t.merge_key.pmid, '') AS pmid,
       CASE WHEN t.merge_key.arxiv IN ('arXiv:', '') THEN NULL
            ELSE t.merge_key.arxiv END AS arxiv,
       NULLIF(t.merge_key.title_author, '') AS title_author,
       CAST(get(filter(t.ids, x -> x.namespace = 'mag').id, 0) AS BIGINT) AS mag_id
FROM t
LEFT JOIN openalex.works.location_work_ids r
  ON  t.provenance = r.provenance
  AND t.native_id_namespace = r.native_id_namespace
  AND t.native_id = r.native_id
WHERE r.native_id IS NULL OR r.work_id IS NULL

## Key lookups — one definition, late-bound
Temporary views so phase 1 and phase 2 share identical guard logic while seeing the
map as of their own execution time (phase 1 pre-insert, phase 2 post-insert). Each is
semi-joined to pending's keys so the map is probed, not fully aggregated. `ta` exposes
the distinct-id count instead of hiding it in a HAVING, so blocked keys (> 3 works)
are distinguishable from unseen keys.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW map_doi_lookup AS
SELECT regexp_replace(m.doi, '[^a-zA-Z0-9\./-]', '') AS k, MIN(m.id) AS v
FROM openalex.works.work_id_map m
LEFT SEMI JOIN (SELECT DISTINCT doi_clean FROM openalex.works.location_work_ids_pending WHERE doi_clean IS NOT NULL) pk
  ON regexp_replace(m.doi, '[^a-zA-Z0-9\./-]', '') = pk.doi_clean
WHERE m.doi IS NOT NULL AND m.doi <> ''
GROUP BY regexp_replace(m.doi, '[^a-zA-Z0-9\./-]', '')

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW map_pmid_lookup AS
SELECT m.pmid AS k, MIN(m.id) AS v
FROM openalex.works.work_id_map m
LEFT SEMI JOIN (SELECT DISTINCT pmid FROM openalex.works.location_work_ids_pending WHERE pmid IS NOT NULL) pk
  ON m.pmid = pk.pmid
WHERE m.pmid IS NOT NULL AND m.pmid <> ''
GROUP BY m.pmid

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW map_arxiv_lookup AS
SELECT m.arxiv AS k, MIN(m.id) AS v
FROM openalex.works.work_id_map m
LEFT SEMI JOIN (SELECT DISTINCT arxiv FROM openalex.works.location_work_ids_pending WHERE arxiv IS NOT NULL) pk
  ON m.arxiv = pk.arxiv
WHERE m.arxiv IS NOT NULL AND m.arxiv NOT IN ('arXiv:', '')
GROUP BY m.arxiv

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW map_ta_lookup AS
SELECT m.title_author AS k, MIN(m.id) AS v, COUNT(DISTINCT m.id) AS n_ids
FROM openalex.works.work_id_map m
LEFT SEMI JOIN (SELECT DISTINCT title_author FROM openalex.works.location_work_ids_pending WHERE title_author IS NOT NULL) pk
  ON m.title_author = pk.title_author
WHERE m.title_author IS NOT NULL AND m.title_author <> ''
GROUP BY m.title_author

## Resolve — phase 1 (against the pre-run map)
Strict tier precedence doi → pmid → arxiv → title_author over works that already
exist. Guards: title_author length > 20 and ≤ 3 distinct ids per key. Legacy adoption
(mag id, pmh crosswalk) applies only when no tier resolves but the anchor is mintable.
`established_conflict` marks strong-tier disagreement between existing works (the
bridge class). `routing_key_unseen` marks anchors whose strongest key is new to the
map — resolved ones become alias rows; unresolved ones become fresh mints.

In [ ]:
CREATE OR REPLACE TABLE openalex.works.location_work_ids_verdicts_stage
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
WITH pmh_mapping AS (
  SELECT LOWER(pmh_id) AS pmh_id, MIN(work_id) AS legacy_work_id
  FROM openalex.works_poc.work_id_to_pmh_id_final
  GROUP BY LOWER(pmh_id)
),
rs AS (
  SELECT pd.provenance, pd.native_id_namespace, pd.native_id, pd.is_retry,
         pd.doi, pd.doi_clean, pd.pmid, pd.arxiv, pd.title_author,
         pd.mag_id,
         d.v AS dv, p.v AS pv, a.v AS av,
         CASE WHEN LENGTH(pd.title_author) > 20 AND ta.n_ids <= 3 THEN ta.v END AS tav,
         (ta.k IS NOT NULL AND ta.n_ids > 3) AS ta_blocked,
         SIZE(ARRAY_DISTINCT(FILTER(ARRAY(d.v, p.v, a.v), x -> x IS NOT NULL))) > 1
           AS established_conflict
  FROM openalex.works.location_work_ids_pending pd
  LEFT JOIN map_doi_lookup d   ON pd.doi_clean = d.k
  LEFT JOIN map_pmid_lookup p  ON pd.pmid = p.k
  LEFT JOIN map_arxiv_lookup a ON pd.arxiv = a.k
  LEFT JOIN map_ta_lookup ta   ON pd.title_author = ta.k
),
scored AS (
  SELECT rs.*,
         COALESCE(rs.dv, rs.pv, rs.av, rs.tav) AS resolved_id,
         CASE WHEN rs.dv IS NOT NULL THEN 'doi'
              WHEN rs.pv IS NOT NULL THEN 'pmid'
              WHEN rs.av IS NOT NULL THEN 'arxiv'
              WHEN rs.tav IS NOT NULL THEN 'title_author' END AS resolved_source,
         (rs.doi IS NOT NULL OR rs.pmid IS NOT NULL OR rs.arxiv IS NOT NULL
          OR (LENGTH(rs.title_author) > 20 AND NOT COALESCE(rs.ta_blocked, FALSE)))
           AS mintable,
         CASE WHEN rs.doi IS NOT NULL THEN rs.dv IS NULL
              WHEN rs.pmid IS NOT NULL THEN rs.pv IS NULL
              WHEN rs.arxiv IS NOT NULL THEN rs.av IS NULL
              WHEN rs.title_author IS NOT NULL THEN rs.tav IS NULL
              ELSE FALSE END AS routing_key_unseen
  FROM rs
)
SELECT s.provenance, s.native_id_namespace, s.native_id, s.is_retry,
       s.doi, s.doi_clean, s.pmid, s.arxiv, s.title_author,
       s.established_conflict, s.mintable, s.routing_key_unseen,
       CASE
         WHEN s.resolved_id IS NOT NULL THEN s.resolved_id
         WHEN s.mintable AND s.provenance = 'mag' AND s.mag_id IS NOT NULL THEN s.mag_id
         WHEN s.mintable AND s.provenance IN ('repo', 'repo_backfill')
              AND pm.legacy_work_id IS NOT NULL THEN pm.legacy_work_id
       END AS work_id,
       CASE
         WHEN s.resolved_id IS NOT NULL THEN s.resolved_source
         WHEN s.mintable AND s.provenance = 'mag' AND s.mag_id IS NOT NULL THEN 'mag_legacy'
         WHEN s.mintable AND s.provenance IN ('repo', 'repo_backfill')
              AND pm.legacy_work_id IS NOT NULL THEN 'pmh_legacy'
       END AS work_id_source
FROM scored s
LEFT JOIN pmh_mapping pm
  ON s.provenance IN ('repo', 'repo_backfill') AND LOWER(s.native_id) = pm.pmh_id

## Map inserts — rows born with their verdict
Insert order matters and each cell anti-joins the slices earlier cells inserted, so a
key can enter the map exactly once per run:
1. **Aliases** (explicit `id` = the phase-1 verdict): resolved/adopted anchors whose
   strongest key is new — the key is born pointing at the right work.
2. **Fresh mints** (generated `id`): nothing resolved; one insert per tier, each
   skipping keys any earlier slice carries.
3. **Enrichment** (the only UPDATE in normal operation): resolved anchors gift their
   secondary keys to the row they resolved on — single QUALIFY-winner per key, empty
   slots only, `id` never touched.

In [ ]:
INSERT INTO openalex.works.work_id_map
  (id, doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT s.work_id, s.doi, s.pmid, s.arxiv, s.title_author,
       current_date(), current_timestamp()
FROM openalex.works.location_work_ids_verdicts_stage s
WHERE s.work_id IS NOT NULL AND s.routing_key_unseen
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY COALESCE(s.doi_clean, s.pmid, s.arxiv, s.title_author)
  ORDER BY (s.doi IS NULL), (s.pmid IS NULL), (s.arxiv IS NULL), s.native_id
) = 1

In [ ]:
INSERT INTO openalex.works.work_id_map
  (doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT s.doi, s.pmid, s.arxiv, s.title_author,
       current_date(), current_timestamp()
FROM openalex.works.location_work_ids_verdicts_stage s
WHERE s.work_id IS NULL AND s.mintable AND s.doi IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 FROM openalex.works.location_work_ids_verdicts_stage x
    WHERE x.work_id IS NOT NULL AND x.routing_key_unseen
      AND x.doi_clean = s.doi_clean)
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY s.doi_clean
  ORDER BY (s.pmid IS NULL), (s.arxiv IS NULL), (s.title_author IS NULL), s.native_id
) = 1

In [ ]:
INSERT INTO openalex.works.work_id_map
  (doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT s.doi, s.pmid, s.arxiv, s.title_author,
       current_date(), current_timestamp()
FROM openalex.works.location_work_ids_verdicts_stage s
WHERE s.work_id IS NULL AND s.mintable AND s.doi IS NULL AND s.pmid IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 FROM openalex.works.location_work_ids_verdicts_stage x
    WHERE x.pmid = s.pmid
      AND ((x.work_id IS NOT NULL AND x.routing_key_unseen)
           OR (x.work_id IS NULL AND x.mintable AND x.doi IS NOT NULL)))
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY s.pmid
  ORDER BY (s.arxiv IS NULL), (s.title_author IS NULL), s.native_id
) = 1

In [ ]:
INSERT INTO openalex.works.work_id_map
  (doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT s.doi, s.pmid, s.arxiv, s.title_author,
       current_date(), current_timestamp()
FROM openalex.works.location_work_ids_verdicts_stage s
WHERE s.work_id IS NULL AND s.mintable
  AND s.doi IS NULL AND s.pmid IS NULL AND s.arxiv IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 FROM openalex.works.location_work_ids_verdicts_stage x
    WHERE x.arxiv = s.arxiv
      AND ((x.work_id IS NOT NULL AND x.routing_key_unseen)
           OR (x.work_id IS NULL AND x.mintable
               AND (x.doi IS NOT NULL OR x.pmid IS NOT NULL))))
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY s.arxiv
  ORDER BY (s.title_author IS NULL), s.native_id
) = 1

In [ ]:
INSERT INTO openalex.works.work_id_map
  (doi, pmid, arxiv, title_author, created_date, updated_date)
SELECT s.doi, s.pmid, s.arxiv, s.title_author,
       current_date(), current_timestamp()
FROM openalex.works.location_work_ids_verdicts_stage s
WHERE s.work_id IS NULL AND s.mintable
  AND s.doi IS NULL AND s.pmid IS NULL AND s.arxiv IS NULL AND s.title_author IS NOT NULL
  AND NOT EXISTS (
    SELECT 1 FROM openalex.works.location_work_ids_verdicts_stage x
    WHERE x.title_author = s.title_author
      AND ((x.work_id IS NOT NULL AND x.routing_key_unseen)
           OR (x.work_id IS NULL AND x.mintable
               AND (x.doi IS NOT NULL OR x.pmid IS NOT NULL OR x.arxiv IS NOT NULL))))
QUALIFY ROW_NUMBER() OVER (PARTITION BY s.title_author ORDER BY s.native_id) = 1

In [ ]:
MERGE INTO openalex.works.work_id_map AS target
USING (
  SELECT s.doi_clean, s.pmid, s.arxiv, s.title_author
  FROM openalex.works.location_work_ids_verdicts_stage s
  WHERE s.work_id_source = 'doi' AND s.doi IS NOT NULL
    AND (s.pmid IS NOT NULL OR s.arxiv IS NOT NULL OR s.title_author IS NOT NULL)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY s.doi_clean
    ORDER BY (s.pmid IS NULL), (s.arxiv IS NULL), (s.title_author IS NULL), s.native_id
  ) = 1
) AS source
ON regexp_replace(target.doi, '[^a-zA-Z0-9\./-]', '') = source.doi_clean
WHEN MATCHED AND (
  (target.pmid IS NULL AND source.pmid IS NOT NULL) OR
  (target.arxiv IS NULL AND source.arxiv IS NOT NULL) OR
  (target.title_author IS NULL AND source.title_author IS NOT NULL)
)
THEN UPDATE SET
  target.pmid = COALESCE(target.pmid, source.pmid),
  target.arxiv = COALESCE(target.arxiv, source.arxiv),
  target.title_author = COALESCE(target.title_author, source.title_author),
  target.updated_date = current_timestamp()

In [ ]:
MERGE INTO openalex.works.work_id_map AS target
USING (
  SELECT s.pmid, s.arxiv, s.title_author
  FROM openalex.works.location_work_ids_verdicts_stage s
  WHERE s.work_id_source = 'pmid' AND s.pmid IS NOT NULL
    AND (s.arxiv IS NOT NULL OR s.title_author IS NOT NULL)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY s.pmid
    ORDER BY (s.arxiv IS NULL), (s.title_author IS NULL), s.native_id
  ) = 1
) AS source
ON target.pmid = source.pmid
WHEN MATCHED AND (
  (target.arxiv IS NULL AND source.arxiv IS NOT NULL) OR
  (target.title_author IS NULL AND source.title_author IS NOT NULL)
)
THEN UPDATE SET
  target.arxiv = COALESCE(target.arxiv, source.arxiv),
  target.title_author = COALESCE(target.title_author, source.title_author),
  target.updated_date = current_timestamp()

In [ ]:
MERGE INTO openalex.works.work_id_map AS target
USING (
  SELECT s.arxiv, s.title_author
  FROM openalex.works.location_work_ids_verdicts_stage s
  WHERE s.work_id_source = 'arxiv' AND s.arxiv IS NOT NULL AND s.title_author IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY s.arxiv ORDER BY (s.title_author IS NULL), s.native_id
  ) = 1
) AS source
ON target.arxiv = source.arxiv
WHEN MATCHED AND target.title_author IS NULL AND source.title_author IS NOT NULL
THEN UPDATE SET
  target.title_author = source.title_author,
  target.updated_date = current_timestamp()

## Resolve — phase 2 and the verdicts table
Anchors that needed a mint pick up their generated id by re-reading the (late-bound)
lookups, which now include this run's inserts; everything else carries its phase-1
verdict. `converged_alias` counts strongest-key-new anchors bound to existing/legacy
works — duplicates prevented at birth (successor concept to the retired write-back's
`mints_displaced`, recorded in its own column).

In [ ]:
CREATE OR REPLACE TABLE openalex.works.location_work_ids_verdicts
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
SELECT s.provenance, s.native_id_namespace, s.native_id, s.is_retry,
       s.established_conflict,
       (s.work_id IS NOT NULL AND s.routing_key_unseen) AS converged_alias,
       COALESCE(s.work_id, d.v, p.v, a.v,
                CASE WHEN LENGTH(s.title_author) > 20 AND ta.n_ids <= 3 THEN ta.v END)
         AS work_id,
       COALESCE(s.work_id_source,
                CASE WHEN d.v IS NOT NULL THEN 'doi'
                     WHEN p.v IS NOT NULL THEN 'pmid'
                     WHEN a.v IS NOT NULL THEN 'arxiv'
                     WHEN LENGTH(s.title_author) > 20 AND ta.n_ids <= 3
                          AND ta.v IS NOT NULL THEN 'title_author' END) AS work_id_source,
       current_timestamp() AS run_ts
FROM openalex.works.location_work_ids_verdicts_stage s
LEFT JOIN map_doi_lookup d   ON s.work_id IS NULL AND s.doi_clean = d.k
LEFT JOIN map_pmid_lookup p  ON s.work_id IS NULL AND s.pmid = p.k
LEFT JOIN map_arxiv_lookup a ON s.work_id IS NULL AND s.arxiv = a.k
LEFT JOIN map_ta_lookup ta   ON s.work_id IS NULL AND s.title_author = ta.k

In [ ]:
MERGE INTO openalex.works.location_work_ids AS target
USING openalex.works.location_work_ids_verdicts AS source
ON  target.provenance = source.provenance
AND target.native_id_namespace = source.native_id_namespace
AND target.native_id = source.native_id
WHEN MATCHED AND target.work_id IS NULL AND source.work_id IS NOT NULL
THEN UPDATE SET
  target.work_id = source.work_id,
  target.work_id_source = source.work_id_source,
  target.openalex_updated_dt = source.run_ts
WHEN NOT MATCHED THEN INSERT (
  provenance, native_id_namespace, native_id, work_id, work_id_source,
  openalex_created_dt, openalex_updated_dt, seeded_dt, seeded_from
) VALUES (
  source.provenance, source.native_id_namespace, source.native_id,
  source.work_id, source.work_id_source,
  current_date(), source.run_ts, NULL, NULL
)

## Per-run match-quality stats
Computed from the verdicts table. `converged_alias` has its own column (the retired
write-back's `mints_displaced` series ends at the deploy date and stays NULL rather
than silently changing meaning). `bridge_conflicts` counts strong-key disagreement
between existing works. Idempotent on `run_ts`; a run with no pending anchors logs
nothing.

In [ ]:
INSERT INTO openalex.works.location_work_ids_run_stats
  (run_ts, run_dt, anchors_new_resolved, anchors_new_null,
   anchors_retried_resolved, anchors_retried_null,
   src_doi, src_pmid, src_arxiv, src_title_author, src_mag_legacy, src_pmh_legacy,
   minted_range, distinct_works, max_work_fanin, top_fanin_work_id, registry_null_backlog,
   mints_displaced, bridge_conflicts, converged_alias)
WITH f AS (
  SELECT work_id, COUNT(*) AS n
  FROM openalex.works.location_work_ids_verdicts
  WHERE work_id IS NOT NULL
  GROUP BY work_id ORDER BY n DESC LIMIT 1
)
SELECT
  MAX(v.run_ts),
  CAST(MAX(v.run_ts) AS DATE),
  SUM(CASE WHEN NOT v.is_retry AND v.work_id IS NOT NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN NOT v.is_retry AND v.work_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.is_retry AND v.work_id IS NOT NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.is_retry AND v.work_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'doi' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'pmid' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'arxiv' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'title_author' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'mag_legacy' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'pmh_legacy' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id > 6600000000 THEN 1 ELSE 0 END),
  COUNT(DISTINCT v.work_id),
  MAX(f.n),
  MAX(f.work_id),
  (SELECT COUNT(*) - COUNT(work_id) FROM openalex.works.location_work_ids),
  CAST(NULL AS BIGINT),
  SUM(CASE WHEN v.established_conflict THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.converged_alias THEN 1 ELSE 0 END)
FROM openalex.works.location_work_ids_verdicts v
LEFT JOIN f ON TRUE
HAVING MAX(v.run_ts) IS NOT NULL
   AND NOT EXISTS (
     SELECT 1 FROM openalex.works.location_work_ids_run_stats s WHERE s.run_ts = MAX(v.run_ts))

In [ ]:
SELECT format_number(COUNT(*), 0) AS registry_anchors,
       format_number(COUNT(work_id), 0) AS with_work_id
FROM openalex.works.location_work_ids